In [ ]:
import sys
sys.path.insert(0, '/mnt/disk_data/python_libs')

In [2]:
import os
import time
import matplotlib.pyplot as plt

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

Режим картинки на области

In [3]:
import cv2
import os

class Preprocessing:
    pass

Содадим датасет

In [4]:
class SegmentationDataset(Dataset):
  def __init__(self, root_dir, size_w: int, size_h: int, transform_img=None, transform_mask=None):
    self.root_dir = root_dir
    self.size_w = size_w
    self.size_h = size_h
    self.transform_img = transform_img
    self.transform_mask = transform_mask

    self.list_files()

  def list_files(self):
    images_dir = os.path.join(self.root_dir, 'images/')
    masks_dir = os.path.join(self.root_dir, 'masks/')

    images_dir_files = os.listdir(images_dir)
    mask_dir_files = os.listdir(masks_dir)

    image_names = sorted([img for img in images_dir_files if os.path.isfile(images_dir + img)])
    mask_names = sorted([mask for mask in mask_dir_files if os.path.isfile(masks_dir + mask)])
    
    self.images = [images_dir + i for i in image_names]
    self.masks = [masks_dir + i for i in mask_names]

  def __getitem__(self, idx):
    image = Image.open(self.images[idx]).convert("RGB")
    mask = Image.open(self.masks[idx]).convert("L")

    image = image.resize((self.size_w, self.size_h))
    original_img = image.copy()

    if self.transform_img:
      image = self.transform_img(image)
    if self.transform_mask:
      mask = self.transform_mask(mask)

    sample = {'image': image, 'mask': mask, "orig_img": np.array(original_img)}

    return sample

  def __len__(self):
    return len(self.images)


In [5]:
class ToBin:
  def __call__(self, image):
    res = image.clone()
    res[image > 0] = 1
    return res

In [6]:
base_model = resnet18(weights=ResNet18_Weights.DEFAULT)

In [7]:
base_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Модель

In [8]:
class ResNetUNet(nn.Module):
  def __init__(self, freeze=True):
    super(ResNetUNet, self).__init__()

    base_model = resnet18(weights=ResNet18_Weights.DEFAULT)

    self.left_conv0 = nn.Sequential(base_model.conv1, base_model.bn1, base_model.relu) # 64 канала
    self.left_pool = base_model.maxpool
    self.left_conv1 = base_model.layer1 # 64 канала
    self.left_conv2 = base_model.layer2 # 128 каналов
    self.left_conv3 = base_model.layer3 # 256 каналов
    self.left_conv4 = base_model.layer4 # 512 каналов (bottleneck)

    if freeze:
      for param in [self.left_conv0, self.left_conv1, self.left_conv2, self.left_conv3, self.left_conv4]:
          for p in param.parameters():
              p.requires_grad = False

    self.right_conv1 = nn.Sequential(
      nn.Conv2d(768, 256, (3, 3), padding=1),
      nn.ReLU(),
      nn.Conv2d(256, 128, (3, 3), padding=1),
      nn.ReLU(),
    )
    
    self.right_conv2 = nn.Sequential(
      nn.Conv2d(256, 128, (3, 3), padding=1),
      nn.ReLU(),
      nn.Conv2d(128, 64, (3, 3), padding=1),
      nn.ReLU()
    )

    self.right_conv3 = nn.Sequential(
      nn.Conv2d(128, 64, (3, 3), padding=1),
      nn.ReLU(),
      nn.Conv2d(64, 32, (3, 3), padding=1),
      nn.ReLU()
    )

    self.right_conv4 = nn.Sequential(
      nn.Conv2d(96, 32, (3, 3), padding=1),
      nn.ReLU(),
      nn.Conv2d(32, 1, (3, 3), padding=1),
      nn.Sigmoid()
    )

    self.up = nn.UpsamplingBilinear2d(scale_factor=2)
    self.pool = nn.MaxPool2d(kernel_size=2)

    
  def forward(self, x):
    left0 = self.left_conv0(x) # (H/2 x W/2 x 64)
    x = self.left_pool(left0) # (H/4 x W/4 x 64)
    left1 = self.left_conv1(x) # (H/4 x W/4 x 64)
    left2 = self.left_conv2(left1) # (H/8 x W/8 x 128)
    left3 = self.left_conv3(left2) # (H/16 x W/16 x 256)
    left4 = self.left_conv4(left3) # (H/32 x W/32 x 512)

    up1 = self.up(left4) # (H/16 x W/16 x 512)
    concat1 = torch.cat((up1, left3), 1) # (H/16 x W/16 x 768)
    right1 = self.right_conv1(concat1) # (H/16 x W/16 x 128)

    up2 = self.up(right1) # (H/8 x H/8 x 128)
    concat2 = torch.cat((up2, left2), 1) # (H/8 x H/8 x 256)
    right2 = self.right_conv2(concat2) # (H/8 x H/8 x 64)

    up3 = self.up(right2) # (H/4 x H/4 x 64)
    concat3 = torch.cat((up3, left1), 1) # (H/4 x H/4 x 128)
    right3 = self.right_conv3(concat3) # (H/4 x H/4 x 32)

    up4 = self.up(right3) # (H/2 x H/2 x 32)
    concat4 = torch.cat((up4, left0), 1) # (H/2 x H/2 x 96)
    right4 = self.right_conv4(concat4) # (H/2, W/2, 1)

    output = self.up(right4) # (H x W x 1)
    return output

Эпоха

In [9]:
def make_epoch(model, criterion, dataloader, optimizer, device, alpha, metrics, epoch, num_epochs, fieldnames, loss_list):
    # Обучение
    since = time.time()
    print('Epoch {}/{}'.format(epoch, num_epochs))
    print('-' * 10)

    epoch_summary = {a: [0] for a in fieldnames}
    epoch_summary['epoch'] = epoch
    running_loss = {f: [] for f in fieldnames[1:]}

    model.train()
    
    for sample in dataloader['train']:
      inputs = sample['image'].to(device)
      masks = sample['mask'].to(device)
      optimizer.zero_grad()
      outputs = model(inputs)
      loss = criterion(outputs, masks)
      loss.backward()
      optimizer.step()

      running_loss['Train_loss'].append(loss.item())      
      y_pred = (outputs.detach().cpu().numpy().ravel() > alpha).astype('uint8')
      y_true = masks.detach().cpu().numpy().ravel().astype('uint8')
      for name, metric in metrics.items():
        running_loss[f'Train_{name}'].append(metrics[name](y_true, y_pred))

    # Тестирование
    model.eval()
    with torch.no_grad():
      for sample in dataloader['test']:
        inputs = sample['image'].to(device)
        masks = sample['mask'].to(device)
        outputs = model(inputs)
        loss = criterion(outputs, masks)
        
        running_loss['Test_loss'].append(loss.item())
        y_pred = (outputs.detach().cpu().numpy().ravel() > alpha).astype('uint8')
        y_true = masks.detach().cpu().numpy().ravel().astype('uint8')
        for name, metric in metrics.items():
          running_loss[f'Test_{name}'].append(metric(y_true, y_pred))
    
    for field in fieldnames[1:]:
      epoch_summary[field] = np.mean(running_loss[field])
    loss_list.append(epoch_summary)

    print(epoch_summary)
    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))

Функция обучения модели

In [10]:
def train_model(model, criterion, dataloader, optimizer, alpha, metrics, num_epochs, loss_list):
  fieldnames = ['epoch', 'Train_loss', 'Test_loss'] + [f'Train_{m}' for m in metrics.keys()] + [f'Test_{m}' for m in metrics.keys()]

  # Кидаем на устройство
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  print(device)
  model.to(device)

  for epoch in range(len(loss_list) + 1, num_epochs + 1):
    make_epoch(model, criterion, dataloader, optimizer, device, alpha, metrics, epoch, num_epochs, fieldnames, loss_list)
  return model

Обучение

In [11]:
'''
team006@team006:~/GeoAIprivate/dataset/train/images$ i=1; for f in *; do mv "$f" "img_$i.png"; ((i++)); done
team006@team006:~/GeoAIprivate/dataset/train/images$ cd ..
team006@team006:~/GeoAIprivate/dataset/train$ cd masks
team006@team006:~/GeoAIprivate/dataset/train/masks$ i=1; for f in *; do mv "$f" "img_$i.png"; ((i++)); done
team006@team006:~/GeoAIprivate/dataset/train/masks$
'''


ROOT = '/home/team006/GeoAIprivate/'
ROOT = '/mnt/disk_data/GeoAIprivate/'
ROOT = 'C:/Users/ASUS/source/repos/GeoAIprivate-main/'

BATCH_SIZE = 4
MAX_EPOCH = 35

size_w = 2272
size_h = 1704

size_w = 224
size_h = 224

size_w -= size_w%32
size_h -= size_h%32

transfrom_image = transforms.Compose([transforms.Resize((size_w, size_h)), transforms.ToTensor(),
                                      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

transfrom_mask = transforms.Compose([transforms.Resize((size_w, size_h)), transforms.ToTensor(), ToBin()])

train_ds = SegmentationDataset(ROOT + "ResNetUNet-dataset/train/", size_w, size_h, transfrom_image, transfrom_mask)
#train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

test_ds = SegmentationDataset(ROOT + "ResNetUNet-dataset/test/", size_w, size_h, transfrom_image, transfrom_mask)
#test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

dataLoader = {"train": train_dl, "test": test_dl}
model = ResNetUNet()
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
alpha = 0.3
metrics = {'accuracy': accuracy_score}
loss_list = []
train_model(model, criterion, dataLoader, optimizer, alpha, metrics, MAX_EPOCH, loss_list)

cpu
Epoch 1/35
----------
{'epoch': 1, 'Train_loss': 0.6725515458318923, 'Test_loss': 0.6484197378158569, 'Train_accuracy': 0.3357265757865646, 'Test_accuracy': 0.3277870230123299}
Training complete in 0m 10s
Epoch 2/35
----------
{'epoch': 2, 'Train_loss': 0.6284058160252042, 'Test_loss': 0.6280319392681122, 'Train_accuracy': 0.4556267493976757, 'Test_accuracy': 0.514626846832483}
Training complete in 0m 10s
Epoch 3/35
----------
{'epoch': 3, 'Train_loss': 0.611279759142134, 'Test_loss': 0.6148448586463928, 'Train_accuracy': 0.5760877267573696, 'Test_accuracy': 0.386029509460034}
Training complete in 0m 9s
Epoch 4/35
----------
{'epoch': 4, 'Train_loss': 0.6046935982174344, 'Test_loss': 0.5922509133815765, 'Train_accuracy': 0.43271056252362056, 'Test_accuracy': 0.5724350286989797}
Training complete in 0m 9s
Epoch 5/35
----------
{'epoch': 5, 'Train_loss': 0.5976379248831007, 'Test_loss': 0.5913481712341309, 'Train_accuracy': 0.6227187706679894, 'Test_accuracy': 0.6502303558142006}
Tra

ResNetUNet(
  (left_conv0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (left_pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (left_conv1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tr

In [12]:
def generate_mask(model, dataloader, save_dir, device):
    model.eval()
    with torch.no_grad():
        for i, sample in enumerate(dataloader):
            inputs = sample['image'].to(device)
            outputs = model(inputs)
            originals = sample['orig_img']
            true_mask = sample['mask']
            
            for j in range(outputs.size(0)):
                mask = (outputs[j].squeeze().cpu().numpy() > alpha).astype('uint8') * 255
                Image.fromarray(mask).save(f"{save_dir}/output_{i * dataloader.batch_size + j}.png")
                Image.fromarray(originals[j].numpy().astype('uint8')).save(f"{save_dir}/img_{i * dataloader.batch_size + j}.png")
                true_mask_np = true_mask[j].squeeze().cpu().numpy().astype('uint8')
                if true_mask_np.max() <= 1:
                    true_mask_np = true_mask_np * 255
                Image.fromarray(true_mask_np).save(f"{save_dir}/mask_{i * dataloader.batch_size + j}.png")

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
generate_mask(model, test_dl, "test_result", device)

In [14]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
generate_mask(model, train_dl, "train_result", device)